# Teil 1 - Use Case Analyse

## 1. Datensatz-Beschreibung: Was wurde erfasst, von wem und warum?

Laut der offiziellen Beschreibung auf Kaggle ("E-Commerce Shipping Data", [kaggle.com/datasets/prachi13/customer-analytics](https://www.kaggle.com/datasets/prachi13/customer-analytics)) stammt der Datensatz von einem **internationalen E-Commerce-Unternehmen, das Elektronikprodukte verkauft**. Das Unternehmen wollte anhand seiner Kundendatenbank wichtige Erkenntnisse ("key insights") über sein Geschäft gewinnen, insbesondere im Zusammenhang mit der Lieferzuverlässigkeit.

Der Datensatz enthält **10.999 Bestelldatensätze** (Train.csv, 12 Spalten). Erfasst wurden pro Bestellung u. a.:

- **Logistik-Merkmale**: Lagerblock (`Warehouse_block`: A–F), Versandart (`Mode_of_Shipment`: Flight, Ship, Road), Produktgewicht (`Weight_in_gms`)
- **Kundenbezogene Merkmale**: Anzahl Anrufe beim Kundenservice zur Sendungsanfrage (`Customer_care_calls`), Kundenbewertung 1–5 (`Customer_rating`), Geschlecht (`Gender`), Anzahl früherer Käufe (`Prior_purchases`)
- **Produkt-/Bestellmerkmale**: Produktpreis in USD (`Cost_of_the_Product`), Produktwichtigkeit (`Product_importance`: low/medium/high), gewährter Rabatt (`Discount_offered`)
- **Zielgröße**: ob die Lieferung pünktlich ankam (`Reached.on.Time_Y.N`, 1 = nicht pünktlich, 0 = pünktlich)

Erfasst wurde der Datensatz vom **operativen/logistischen System des Händlers** (Auftragsverwaltung, Versand- und CRM-Systeme), das bei jeder Bestellung automatisch Versand-, Produkt- und Kundendaten mitschreibt. Der Zweck der Erfassung ist die **Nachverfolgung der Lieferzuverlässigkeit**: Das Unternehmen möchte verstehen, welche Faktoren (Versandart, Gewicht, Rabatt, Kundenverhalten etc.) mit verspäteten Lieferungen zusammenhängen, um daraus Optimierungspotenzial für die Lieferkette abzuleiten.

## 2. Business-Fragestellung

**Kann das Unternehmen anhand von Bestell-, Versand- und Kundenmerkmalen einer neuen Sendung frühzeitig erkennen, ob diese sich verspäten wird, um durch gezieltes Gegensteuern (z. B. Versandart wechseln, Kunden proaktiv informieren) Reklamationen zu vermeiden und die Kundenzufriedenheit zu erhöhen?**

## 3. Art des Problems

Es handelt sich um ein **Klassifikationsproblem**, genauer eine **binäre Klassifikation**.

**Begründung:** Die Zielvariable `Reached.on.Time_Y.N` nimmt nur zwei diskrete Werte an (0 = pünktlich, 1 = nicht pünktlich). Es soll also keine kontinuierliche Zahl vorhergesagt werden, sondern eine Bestellung einer von zwei Klassen zugeordnet werden. 

## 4. Mehrwert

Ein trainiertes Modell könnte einem Unternehmen folgenden Nutzen bringen:

- **Proaktives Risikomanagement**: Bestellungen mit hohem Verspätungsrisiko können frühzeitig identifiziert werden, sodass der Kunde informiert oder die Sendung priorisiert bearbeitet werden kann.
- **Optimierung der Lieferkette**: Erkenntnisse darüber, welche Faktoren (z. B. Versandart, Gewicht, Lagerblock) Verspätungen begünstigen, helfen bei der Prozessverbesserung (z. B. Wechsel der Versandart für kritische Produkte).
- **Realistischere Lieferzeitangaben**: Realistischere Lieferzeitangaben erhöhen die Kundenzufriedenheit und senken Support-Anfragen (die Anzahl der `Customer_care_calls` zeigt, dass Lieferunsicherheit bereits heute Supportaufwand verursacht).
- **Kosteneinsparung**: Weniger Reklamationen, Rückerstattungen und Eilzustellungen durch gezieltes Gegensteuern bei Risikobestellungen.
- **Datengetriebene Entscheidungen**: Das Modell liefert eine quantitative Grundlage für strategische Entscheidungen (z. B. Auswahl von Logistikpartnern oder Warehouse-Standorten).

## 5. Einschränkungen

- **Datenqualität und Repräsentativität**: Der Datensatz stammt von einem bestimmten Händler und Zeitraum; das Modell könnte auf andere Märkte, Saisons (z. B. Weihnachtsgeschäft) oder Unternehmen schlecht übertragbar sein.
- **Fehlende Kontextvariablen**: Externe Einflüsse wie Wetter, Zoll, Streiks oder regionale Infrastruktur werden nicht erfasst, obwohl sie Lieferverspätungen stark beeinflussen können.
- **Fairness/Bias**: Merkmale wie `Gender` sollten kritisch geprüft werden, damit das Modell keine unfairen oder diskriminierenden Muster lernt (z. B. unterschiedliche Behandlung von Kundengruppen ohne sachlichen Grund).
- **Fehlinterpretation von Korrelation als Kausalität**: Ein Zusammenhang zwischen einem Merkmal (z. B. Rabatt) und Verspätung bedeutet nicht zwangsläufig eine ursächliche Beziehung.
- **Fehlklassifikationen und Konsequenzen**: Falsch-negative Vorhersagen (angeblich pünktlich, tatsächlich verspätet) könnten dazu führen, dass Kunden nicht rechtzeitig informiert werden; die Kosten von Fehlern in beide Richtungen sollten bei der Modellbewertung berücksichtigt werden.

# Teil 2 - Datenprozessierung

## 2.1 Import und Überblick

- **Customer_rating:** 
1 ist die niedrigste Bewertung (am schlechtesten), 5 die höchste (am besten).
- **Cost_of_the_Product:** 
Preis des Produkts in US-Dollar.
- **Reached.on.Time_Y.N:** 
1 gibt an, dass das Produkt NICHT pünktlich angekommen ist, und 0, dass es pünktlich angekommen ist.

In [45]:
import pandas as pd

df = pd.read_csv("Train.csv")
df.shape
df.info()
df.describe()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 10999 entries, 0 to 10998
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   ID                   10999 non-null  int64
 1   Warehouse_block      10999 non-null  str  
 2   Mode_of_Shipment     10999 non-null  str  
 3   Customer_care_calls  10999 non-null  int64
 4   Customer_rating      10999 non-null  int64
 5   Cost_of_the_Product  10999 non-null  int64
 6   Prior_purchases      10999 non-null  int64
 7   Product_importance   10999 non-null  str  
 8   Gender               10999 non-null  str  
 9   Discount_offered     10999 non-null  int64
 10  Weight_in_gms        10999 non-null  int64
 11  Reached.on.Time_Y.N  10999 non-null  int64
dtypes: int64(8), str(4)
memory usage: 1.0 MB


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


Erkenntnis: Es hat keine Null-Werte. 

## 2.2 Datenqualität prüfen und bereinigen

Da es anhand df.describe keine Nullwerte gibt, müssen keine fehlende Werte identifieziert werden.
Über verschiedene Spalten sind die Stringdaten in unterschiedliche String Cases erfasst worden.
Diese Spalten wurden normalisiert und in Grossbuchstaben transformiert.


------------


Duplikate prüfen. 
Die ID Spalte wird bewusst ausgeschlossen, da diese Unique sind.

In [46]:
print(df.drop(columns=["ID"]).duplicated().sum())


0


Es sind keine Duplikaten vorhanden.

---------------------

Häufigkeit der pünktlich/nich pünktlich angekommenen Lieferungen

In [47]:
s=df["Reached.on.Time_Y.N"]
print(s.value_counts())

Reached.on.Time_Y.N
1    6563
0    4436
Name: count, dtype: int64


---

Ausreisser und unplausible Daten für 
- Cost_of_the_Product
- Discount_offered
- Weight_in_gms


In [48]:
Q1=df["Cost_of_the_Product"].quantile(0.25)
Q3=df["Cost_of_the_Product"].quantile(0.75)
IQR=Q3-Q1

ug = Q1 - 1.5 * IQR
og = Q3 + 1.5 * IQR

ausreisser = df[
    (df["Cost_of_the_Product"] < ug) | 
    (df["Cost_of_the_Product"] > og)]

print(f'Ausreisser: {len(ausreisser)}')
print(ug, og)

Ausreisser: 0
46.0 374.0


In [49]:
mean = df["Cost_of_the_Product"].mean()
std = df["Cost_of_the_Product"].std()

df["z_score"] = (df["Cost_of_the_Product"] - mean) / std

ausreisser_z = df[df["z_score"].abs() > 3]

print(f'Ausreisser: {len(ausreisser_z)}')


Ausreisser: 0


Es gibt keine Ausreisse für Kosten

In [50]:
Q1=df["Weight_in_gms"].quantile(0.25)
Q3=df["Weight_in_gms"].quantile(0.75)
IQR=Q3-Q1

ug = Q1 - 1.5 * IQR
og = Q3 + 1.5 * IQR

ausreisser = df[
    (df["Weight_in_gms"] < ug) | 
    (df["Weight_in_gms"] > og)] 

print(f'Ausreisser: {len(ausreisser)}')
print(ug, og)

Ausreisser: 0
-2976.25 9865.75


Es gibt keine Ausreisse für Gewicht

In [51]:
Q1=df["Discount_offered"].quantile(0.25)
Q3=df["Discount_offered"].quantile(0.75)
IQR=Q3-Q1

ug = Q1 - 1.5 * IQR
og = Q3 + 1.5 * IQR

ausreisser = df[
    (df["Discount_offered"] < ug) | 
    (df["Discount_offered"] > og)]

print(f'Ausreisser: {len(ausreisser)}')
print(ug, og)


Ausreisser: 2209
-5.0 19.0


Es gibt 2209 Ausreisse bei Rabatten

In [52]:
mean = df["Discount_offered"].mean()
std = df["Discount_offered"].std()

df["z_score"] = (df["Discount_offered"] - mean) / std

ausreisser_z = df[df["z_score"].abs() > 3]

print(f'Ausreisser: {len(ausreisser_z)}')

Ausreisser: 181


Problem beim Z-Score: Er basiert auf den Mittelwert und Standardabweichung im gegensatz zu der IQR Methode die auf den Median basiert.

----

Zeichenfehler und inkonsistente Werte

In [53]:
print(df['Mode_of_Shipment'].value_counts())
print(df['Product_importance'].value_counts())
print(df['Gender'].value_counts())
print(df['Warehouse_block'].value_counts())



Mode_of_Shipment
Ship      7462
Flight    1777
Road      1760
Name: count, dtype: int64
Product_importance
low       5297
medium    4754
high       948
Name: count, dtype: int64
Gender
F    5545
M    5454
Name: count, dtype: int64
Warehouse_block
F    3666
D    1834
A    1833
B    1833
C    1833
Name: count, dtype: int64


In [54]:
df['Mode_of_Shipment']=df['Mode_of_Shipment'].str.upper()
df['Product_importance']=df['Product_importance'].str.upper()
df['Gender']=df['Gender'].str.upper()
df['Warehouse_block']=df['Warehouse_block'].str.upper()

print(df['Mode_of_Shipment'].value_counts())
print(df['Product_importance'].value_counts())
print(df['Gender'].value_counts())
print(df['Warehouse_block'].value_counts())


Mode_of_Shipment
SHIP      7462
FLIGHT    1777
ROAD      1760
Name: count, dtype: int64
Product_importance
LOW       5297
MEDIUM    4754
HIGH       948
Name: count, dtype: int64
Gender
F    5545
M    5454
Name: count, dtype: int64
Warehouse_block
F    3666
D    1834
A    1833
B    1833
C    1833
Name: count, dtype: int64


Die Daten sind in Ordnung. Die Daten wurden normaliesiert und einheitlich auf Uppercase bereinigt. 

## 2.3 Explorative Datenanalyse (EDA)